In [0]:
asset_df = spark.read.format("delta").load("/Volumes/telecom_catalog/default/bronze/Asset_config/")

In [0]:
display(asset_df)

In [0]:
asset_df.printSchema()

## Data Profiling

In [0]:
print(asset_df.count())

print(asset_df.count() - asset_df.dropDuplicates().count())

asset_df.groupBy("device_id").count().filter("count > 1").show()

from pyspark.sql.functions import *

display(
    asset_df.select([
        count(when(col(c).isNull(), c)).alias(c)
        for c in asset_df.columns
    ])
)

In [0]:
asset_df.filter(col("device_id").isin("DVC_101", "DVC_120")).display()

In [0]:
silver_df = asset_df.dropDuplicates(["device_id"])

## Handling Nulls

In [0]:
silver_df = silver_df.fillna({
    "firmware_version": "Unknown",
    "maintenance_status": "Unknown",
    "rack_location": "Unassigned",
    "sla_level": "Unassigned"
    
})

# Business Rule Validation

## 1. Valid Asset Types

In [0]:
valid_asset = [
    "Router",
    "Switch",
    "Firewall",
    "Server",
    "Load Balancer"
]

## 2. Valid SLA


In [0]:
valid_sla = [
    "Gold",
    "Silver",
    "Platinum"
]

## 3. Valid Maintenance Status


In [0]:
valid_status = [
    "Active",
    "Maintenance Due",
    "Deprecated"
]

## 4. Create Validation Flag

In [0]:
from pyspark.sql.functions import *

silver_df = silver_df.withColumn(
    "is_valid",
    (
        col("asset_type").isin(valid_asset)
        & col("sla_level").isin(valid_sla)
        & col("maintenance_status").isin(valid_status)
    )
)

# Feature Engineering

## 1. Device Age

In [0]:
from pyspark.sql.functions import datediff, current_date

silver_df = silver_df.withColumn(
    "device_age_days",
    datediff(current_date(), col("install_date"))
)

## 2. Device Age Category

In [0]:
from pyspark.sql.functions import when

silver_df = silver_df.withColumn(
    "device_age_category",
    when(col("device_age_days") < 365, "New")
    .when(col("device_age_days") < 730, "Medium")
    .otherwise("Old")
)

## 3. SLA Priority

In [0]:
silver_df = silver_df.withColumn(
    "sla_priority",
    when(col("sla_level") == "Platinum", 1)
    .when(col("sla_level") == "Gold", 2)
    .otherwise(3)
)

## 4. Maintenance Required


In [0]:
silver_df = silver_df.withColumn(
    "maintenance_required",
    when(col("maintenance_status") == "Maintenance Due", True)
    .otherwise(False)
)

## 5. Audit Column


In [0]:
from pyspark.sql.functions import current_timestamp

silver_df = silver_df.withColumn(
    "silver_load_time",
    current_timestamp()
)

In [0]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("telecom_catalog.default.silver_device_master")
)